# 收益模型：联合拟合、预测与更新

本单元接收已构造的时间正序收益观测，支持 ARMA-GARCH、ARMA-EGARCH、ARMA-GJR-GARCH、ARMA-FIGARCH、ARMA-Component GARCH、ARFIMA-GARCH 六组。采样与去季节由 b03、c01、c02 完成；模型不重新采样或切日，也不在内部取 log、做整数差分或指数还原输入。

## 模型定义与估计目标

均值采用 $\Phi(L)(1-L)^d(r_t-\mu)=\Theta(L)\epsilon_t$，其中 $\Phi(L)=1-\sum\phi_iL^i$，$\Theta(L)=1+\sum\theta_jL^j$。ARMA 的 d 为 0；ARFIMA 的 d 与其他参数联合估计，限制于 (-0.5,0.5)，调用者可指定更窄区间。mean.mu 是该表示中的中心水平，不是 AR 方程截距。include_mean=False 时固定为 0。

全训练序列使用一次高斯准最大似然联合估计：

$$\mathcal L=\frac12\sum_t\left[\log(2\pi)+\log h_t+\epsilon_t^2/h_t\right].$$

每次候选参数都重新计算均值残差和方差递推。函数不先分别拟合两个模型再组合，不执行自动选阶；调用者选择模型和阶数，也可在训练范围内自行选择后调用相同函数。估计准则参考 [arch 正态似然定义](https://arch.readthedocs.io/en/latest/univariate/generated/generated/arch.univariate.Normal.loglikelihood.html)，联合均值表示及 Component 定义参考 [rugarch 方法说明](https://stat.ethz.ch/CRAN/web/packages/rugarch/vignettes/Introduction_to_the_rugarch_package.pdf)，平稳系数变换使用 [statsmodels Monahan 变换](https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.tools.constrain_stationary_univariate.html)。

## 方差形式与约束

- GARCH：$h_t=\omega+\sum\alpha_i\epsilon_{t-i}^2+\sum\beta_jh_{t-j}$。系数非负，持久性严格小于 1。
- GJR：增加 $\sum\gamma_i\epsilon_{t-i}^2I(\epsilon_{t-i}<0)$。允许有效的负 gamma；分别保证正、负冲击系数非负，并限制对称 QML 下的持久性 $\sum\alpha+\frac12\sum\gamma+\sum\beta<1$。
- EGARCH：$\log h_t=\omega+\sum\alpha_i(|z_{t-i}|-\sqrt{2/\pi})+\sum\gamma_iz_{t-i}+\sum\beta_j\log h_{t-j}$。log 方差 AR 部分要求稳定，alpha、gamma 可正可负。
- FIGARCH：采用截断 ARCH 无穷表示。p、q 各支持 0 或 1；$0<d<1$，$0\le\phi\le(1-d)/2$，$0\le\beta\le d+\phi$。使用其专用正值条件，不将普通 GARCH 的持久性条件套在 FIGARCH 上。具体截断长度由 fractional_memory 指定。[arch FIGARCH 定义](https://arch.readthedocs.io/en/latest/univariate/generated/arch.univariate.FIGARCH.html)
- Component：$q_t=\omega+\rho q_{t-1}+\eta(\epsilon_{t-1}^2-h_{t-1})$，$h_t=q_t+\sum\alpha_i(\epsilon_{t-i}^2-q_{t-i})+\sum\beta_j(h_{t-j}-q_{t-j})$。长期递推为 Engle–Lee 一阶形式；短期 ARCH／GARCH 阶数由调用者指定，均至少为 1。采用非负冲击系数、$\sum\alpha+\sum\beta<\rho<1$ 与 $0<\eta<\beta_1$；实际路径中的 h、q 还须严格为正，否则明确失败。短期偏离 $h_t-q_t$ 可以为负，三个量满足恒等关系。

AR 与 MA 使用平稳／可逆参数化。ARFIMA 的分数滤波、FIGARCH 的历史核均使用调用者指定的有限截断；分数模型不是通过单独 Whittle 拟合再接 GARCH。所有模型的预测只针对本序列的下一采样步。

In [21]:
from __future__ import annotations

from copy import deepcopy
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.signal import lfilter
from scipy.special import expit, softmax
from statsmodels.tsa.statespace.tools import constrain_stationary_univariate, unconstrain_stationary_univariate
from arch.univariate import GARCH
from numba import njit

RETURN_MODEL_NAMES = (
    "ARMA-GARCH", "ARMA-EGARCH", "ARMA-GJR-GARCH", "ARMA-FIGARCH",
    "ARMA-Component GARCH", "ARFIMA-GARCH",
)

In [22]:
@njit(cache=False)
def _fractional_return_weights(d, memory):
    weights = np.empty(memory + 1)
    weights[0] = 1.0
    for lag in range(1, memory + 1):
        weights[lag] = weights[lag - 1] * (lag - 1 - d) / lag
    return weights


def _decode_return_parameters(raw_parameters, specification):
    """从优化坐标得到满足对应约束的模型系数和实际递推核。"""
    ar_order, ma_order = specification["mean_order"]
    arch_order, garch_order = specification["volatility_order"]
    asymmetric_order = specification["asymmetric_order"]
    kind = specification["variance_kind"]
    parameters = {}
    cursor = 0
    mu = 0.0
    if specification["include_mean"]:
        mu = float(raw_parameters[cursor]); cursor += 1
        parameters["mean.mu"] = mu
    ar = (constrain_stationary_univariate(np.sinh(raw_parameters[cursor:cursor + ar_order]))
          if ar_order else np.empty(0))
    cursor += ar_order
    ma = (-constrain_stationary_univariate(np.sinh(raw_parameters[cursor:cursor + ma_order]))
          if ma_order else np.empty(0))
    cursor += ma_order
    for lag, coefficient in enumerate(ar, 1):parameters[f"mean.ar[{lag}]"] = float(coefficient)
    for lag, coefficient in enumerate(ma, 1):parameters[f"mean.ma[{lag}]"] = float(coefficient)
    mean_d = 0.0
    if specification["model"] == "ARFIMA-GARCH":
        low, high = specification["fractional_d_bounds"]
        mean_d = low + (high - low) * expit(raw_parameters[cursor]); cursor += 1
        parameters["mean.d"] = float(mean_d)
    mean_weights = (_fractional_return_weights(mean_d, specification["fractional_memory"])
                    if specification["model"] == "ARFIMA-GARCH" else np.ones(1))
    mean_filter = np.convolve(np.r_[1.0, -ar], mean_weights)
    omega = float(raw_parameters[cursor] if kind == 1 else np.exp(raw_parameters[cursor]))
    cursor += 1
    parameters["variance.omega"] = omega
    alpha, gamma, beta = np.empty(0), np.empty(0), np.empty(0)
    figarch_weights = np.empty(0)
    rho, eta = 0.0, 0.0
    if kind in (0, 2):
        size = arch_order + asymmetric_order + garch_order
        weights = softmax(np.r_[raw_parameters[cursor:cursor + size], 0.0])[:-1]
        alpha = weights[:arch_order].copy()
        for lag in range(min(arch_order, asymmetric_order)):alpha[lag] *= 2
        gamma = 2 * weights[arch_order:arch_order + asymmetric_order]
        for lag in range(min(arch_order, asymmetric_order)):gamma[lag] -= alpha[lag]
        beta = weights[arch_order + asymmetric_order:].copy()
    elif kind == 1:
        alpha = raw_parameters[cursor:cursor + arch_order].copy(); cursor += arch_order
        gamma = raw_parameters[cursor:cursor + asymmetric_order].copy(); cursor += asymmetric_order
        beta = (constrain_stationary_univariate(np.sinh(raw_parameters[cursor:cursor + garch_order]))
                if garch_order else np.empty(0))
    elif kind == 3:
        low, high = specification["fractional_d_bounds"]
        d = low + (high - low) * expit(raw_parameters[cursor]); cursor += 1
        phi = 0.5 * (1 - d) * expit(raw_parameters[cursor]) if arch_order else 0.0
        cursor += arch_order
        beta_value = (d + phi) * expit(raw_parameters[cursor]) if garch_order else 0.0
        parameters["variance.d"] = float(d)
        if arch_order:parameters["variance.phi"] = float(phi)
        beta = np.array([beta_value]) if garch_order else np.empty(0)
        fractional_weights = _fractional_return_weights(d, specification["fractional_memory"])
        figarch_weights = -lfilter([1.0, -phi], [1.0, -beta_value], fractional_weights)[1:]
    else:
        rho = float(expit(raw_parameters[cursor])); cursor += 1
        eta_fraction = expit(raw_parameters[cursor]); cursor += 1
        weights = rho * softmax(np.r_[raw_parameters[cursor:cursor + arch_order + garch_order], 0.0])[:-1]
        alpha, beta = weights[:arch_order].copy(), weights[arch_order:].copy()
        eta = float(beta[0] * eta_fraction)
        parameters["variance.rho"], parameters["variance.eta"] = rho, eta
    for name, coefficients in (("alpha", alpha), ("gamma", gamma), ("beta", beta)):
        for lag, coefficient in enumerate(coefficients, 1):parameters[f"variance.{name}[{lag}]"] = float(coefficient)
    return {"parameters": parameters, "mu": mu, "mean_filter": mean_filter, "ma": ma,
            "omega": omega, "alpha": alpha, "gamma": gamma, "beta": beta,
            "rho": rho, "eta": eta, "figarch_weights": figarch_weights}

In [23]:
@njit(cache=False)
def _return_lag(values, presample, position, lag, default):
    source = position - lag
    if source >= 0:return values[source]
    source += len(presample)
    return presample[source] if source >= 0 else default


@njit(cache=False)
def _filter_return_sequence(
    returns, kind, mu, mean_filter, ma, omega, alpha, gamma, beta, rho, eta,
    figarch_weights, initial_variance, initial_component,
    past_returns, past_residuals, past_variances, past_components,
):
    """训练、下一步预测和吸收观测共享同一均值／方差递推。"""
    size = len(returns)
    means, residuals = np.full(size, np.nan), np.full(size, np.nan)
    variances, components = np.full(size, np.nan), np.full(size, np.nan)
    for position in range(size):
        conditional_mean = mu
        for lag in range(1, len(mean_filter)):
            conditional_mean -= mean_filter[lag] * (_return_lag(returns, past_returns, position, lag, mu) - mu)
        for lag in range(1, len(ma) + 1):
            conditional_mean += ma[lag - 1] * _return_lag(residuals, past_residuals, position, lag, 0.0)
        means[position] = conditional_mean
        variance = omega
        component = np.nan
        if kind in (0, 2):
            for lag in range(1, len(alpha) + 1):
                previous_variance = _return_lag(variances, past_variances, position, lag, initial_variance)
                available = position >= lag or len(past_residuals) >= lag - position
                innovation = _return_lag(residuals, past_residuals, position, lag, 0.0)
                squared_innovation = innovation * innovation if available else previous_variance
                variance += alpha[lag - 1] * squared_innovation
            for lag in range(1, len(gamma) + 1):
                previous_variance = _return_lag(variances, past_variances, position, lag, initial_variance)
                available = position >= lag or len(past_residuals) >= lag - position
                innovation = _return_lag(residuals, past_residuals, position, lag, 0.0)
                negative_shock = innovation * innovation * (innovation < 0) if available else 0.5 * previous_variance
                variance += gamma[lag - 1] * negative_shock
            for lag in range(1, len(beta) + 1):
                variance += beta[lag - 1] * _return_lag(variances, past_variances, position, lag, initial_variance)
        elif kind == 1:
            log_variance = omega
            for lag in range(1, max(len(alpha), len(gamma)) + 1):
                available = position >= lag or len(past_residuals) >= lag - position
                if not available:continue
                previous_variance = _return_lag(variances, past_variances, position, lag, initial_variance)
                innovation = _return_lag(residuals, past_residuals, position, lag, 0.0)
                standardized = innovation / np.sqrt(previous_variance)
                if lag <= len(alpha):log_variance += alpha[lag - 1] * (abs(standardized) - np.sqrt(2 / np.pi))
                if lag <= len(gamma):log_variance += gamma[lag - 1] * standardized
            for lag in range(1, len(beta) + 1):
                log_variance += beta[lag - 1] * np.log(_return_lag(variances, past_variances, position, lag, initial_variance))
            variance = np.exp(log_variance)
        elif kind == 3:
            beta_value = beta[0] if len(beta) else 0.0
            variance = omega / (1 - beta_value)
            for lag in range(1, len(figarch_weights) + 1):
                previous_variance = _return_lag(variances, past_variances, position, lag, initial_variance)
                available = position >= lag or len(past_residuals) >= lag - position
                innovation = _return_lag(residuals, past_residuals, position, lag, 0.0)
                squared_innovation = innovation * innovation if available else previous_variance
                variance += figarch_weights[lag - 1] * squared_innovation
        else:
            previous_variance = _return_lag(variances, past_variances, position, 1, initial_variance)
            previous_component = _return_lag(components, past_components, position, 1, initial_component)
            available = position >= 1 or len(past_residuals) >= 1 - position
            innovation = _return_lag(residuals, past_residuals, position, 1, 0.0)
            squared_innovation = innovation * innovation if available else previous_variance
            component = omega + rho * previous_component + eta * (squared_innovation - previous_variance)
            components[position] = component
            variance = component
            for lag in range(1, len(alpha) + 1):
                past_variance = _return_lag(variances, past_variances, position, lag, initial_variance)
                past_component = _return_lag(components, past_components, position, lag, initial_component)
                available = position >= lag or len(past_residuals) >= lag - position
                innovation = _return_lag(residuals, past_residuals, position, lag, 0.0)
                squared_innovation = innovation * innovation if available else past_variance
                variance += alpha[lag - 1] * (squared_innovation - past_component)
            for lag in range(1, len(beta) + 1):
                past_variance = _return_lag(variances, past_variances, position, lag, initial_variance)
                past_component = _return_lag(components, past_components, position, lag, initial_component)
                variance += beta[lag - 1] * (past_variance - past_component)
        if not np.isfinite(variance) or variance <= 0 or (kind == 4 and (not np.isfinite(component) or component <= 0)):
            return means, residuals, variances, components
        variances[position] = variance
        residuals[position] = returns[position] - conditional_mean
        if not np.isfinite(residuals[position]):return means, residuals, variances, components
    return means, residuals, variances, components


def _evaluate_return_sequence(returns, state):
    """将内存状态接入共用递推，报告首个无效结果而不修补路径。"""
    means, residuals, variances, components = _filter_return_sequence(
        np.asarray(returns, dtype=float), state["specification"]["variance_kind"],
        state["mu"], state["mean_filter"], state["ma"], state["omega"],
        state["alpha"], state["gamma"], state["beta"], state["rho"], state["eta"],
        state["figarch_weights"], state["initial_variance"], state["initial_component"],
        state["returns_history"], state["residuals_history"],
        state["variances_history"], state["components_history"],
    )
    valid = np.isfinite(means) & np.isfinite(residuals) & np.isfinite(variances) & (variances > 0)
    if state["specification"]["variance_kind"] == 4:valid &= np.isfinite(components) & (components > 0)
    if not valid.all():
        position = int(np.flatnonzero(~valid)[0])
        return {"status": "failed", "reason": f"序列位置 {position} 的均值、方差或长期成分无效"}
    return {"status": "ok", "reason": "", "mean": means, "residual": residuals,
            "variance": variances, "long_term_variance": components}

## 三个调用接口

`fit_return_model` 接收一个有限值的一维序列；Pandas Series 保留其索引，数组使用顺序索引。model、mean_order=(AR,MA)、volatility_order=(ARCH,GARCH) 必须由调用者提供；GJR／EGARCH 的 asymmetric_order 默认 1，也可显式指定，其他组为 0。ARFIMA／FIGARCH 必须提供正整数 fractional_memory；其他组不使用该参数。没有在函数内部删除缺失行、自动选阶或遍历组合。

初始化可选 backcast 或 sample_variance；initial_variance 可显式覆盖初始方差，Component 还可指定 initial_component。presample 可提供按时间正序的 returns、residuals、variances、components 历史数组；未提供的历史以均值水平、零均值创新期望和初始方差／长期成分初始化。残差缺历史时，方差递推使用相应历史方差作为平方创新期望，GJR 负冲击期望为其一半，EGARCH 中心化冲击期望为 0。初始化和所有拟合数据只来自本次提供的训练输入。

优化器内部仅用训练标准差作线性数值换单位；预测、残差、方差、长期成分和具名系数都还原到调用者的输入尺度。EGARCH 的方差截距按 log 方差平移关系还原。numeric_scale 可由调用者明确指定正数。这里不改变输入的 log／非 log 语义。

fit 返回 status、reason、parameters、specification、state、in_sample、optimization、loglikelihood、next_forecast。训练行全部进入 QML，没有自动丢弃起始段。state 保留下一步所需的有限历史，而非持续累积全部测试序列。

`predict_return_model(state)` 不改变状态；`update_return_model(state, observed_return)` 只吸收一个实现值，返回新 state，保持参数固定且不修改旧 state。返回的 variance_forecast 是采样收益创新的条件方差；RV／BPV 分钟贡献观测的直接建模归 c04。Component 同时返回长期成分和短期偏离。

配置不合法、样本不足、缺失、零变化、优化不收敛、数值或有效性失败均返回 failed 和明确原因。失败拟合不提供可调用 state；失败更新不推进旧状态。只进行一次优化调用，不自动换模型、填值或重试。

In [24]:
def fit_return_model(
    observations, *, model: str, mean_order: tuple[int, int],
    volatility_order: tuple[int, int], asymmetric_order: int | None = None,
    include_mean: bool = True, fractional_memory: int | None = None,
    fractional_d_bounds: tuple[float, float] | None = None,
    initialization: str = "backcast", initial_variance: float | None = None,
    initial_component: float | None = None, presample: dict | None = None,
    numeric_scale: float | None = None, optimizer_options: dict | None = None,
) -> dict:
    """对一个已选模型／阶数进行一次全序列 Gaussian QML 联合估计。"""
    failure = {"status": "failed", "reason": "", "parameters": {}, "state": None}
    try:
        if model not in RETURN_MODEL_NAMES:raise ValueError("model 不属于批准的六组收益模型")
        for name, orders in (("mean_order", mean_order), ("volatility_order", volatility_order)):
            if len(orders) != 2 or any(not isinstance(value, (int, np.integer)) or isinstance(value, (bool, np.bool_)) or value < 0 for value in orders):
                raise ValueError(f"{name} 必须为两个非负整数")
        ar_order, ma_order = tuple(map(int, mean_order))
        arch_order, garch_order = tuple(map(int, volatility_order))
        kind = {"ARMA-GARCH": 0, "ARFIMA-GARCH": 0, "ARMA-EGARCH": 1,
                "ARMA-GJR-GARCH": 2, "ARMA-FIGARCH": 3, "ARMA-Component GARCH": 4}[model]
        asymmetric_order = (1 if kind in (1, 2) else 0) if asymmetric_order is None else asymmetric_order
        if not isinstance(asymmetric_order, (int, np.integer)) or isinstance(asymmetric_order, (bool, np.bool_)) or asymmetric_order < 0:
            raise ValueError("asymmetric_order 必须为非负整数")
        if kind not in (1, 2) and asymmetric_order:raise ValueError("该模型不使用 asymmetric_order")
        if kind == 3 and (arch_order > 1 or garch_order > 1):raise ValueError("FIGARCH 的 p、q 目前各仅支持 0 或 1")
        if kind == 4 and (arch_order < 1 or garch_order < 1):raise ValueError("Component 短期 ARCH、GARCH 阶数均须至少为 1")
        if not isinstance(include_mean, (bool, np.bool_)):raise ValueError("include_mean 必须为布尔值")
        fractional = model == "ARFIMA-GARCH" or kind == 3
        if fractional:
            if not isinstance(fractional_memory, (int, np.integer)) or isinstance(fractional_memory, (bool, np.bool_)) or fractional_memory < 1:
                raise ValueError("分数模型必须显式提供正整数 fractional_memory")
            d_bounds = ((0.0, 1.0) if kind == 3 else (-0.5, 0.5)) if fractional_d_bounds is None else fractional_d_bounds
            low, high = map(float, d_bounds)
            domain_low, domain_high = (0.0, 1.0) if kind == 3 else (-0.5, 0.5)
            if not np.isfinite([low, high]).all() or not domain_low <= low < high <= domain_high:
                raise ValueError("fractional_d_bounds 须为模型允许范围内的递增有限边界")
            d_bounds = (low, high)
        else:
            if fractional_memory is not None or fractional_d_bounds is not None:
                raise ValueError("非分数模型不使用 fractional_memory 或 fractional_d_bounds")
            fractional_memory, d_bounds = 0, None
        if initialization not in ("backcast", "sample_variance"):raise ValueError("initialization 必须为 backcast 或 sample_variance")
        if kind != 4 and initial_component is not None:raise ValueError("只有 Component 使用 initial_component")
        if isinstance(observations, pd.Series):
            values = observations.to_numpy(dtype=float, na_value=np.nan)
            observation_index = observations.index.copy()
        else:
            values = np.asarray(observations, dtype=float)
            observation_index = pd.RangeIndex(len(values)) if values.ndim else pd.RangeIndex(0)
        if values.ndim != 1 or not len(values) or not np.isfinite(values).all():
            raise ValueError("observations 必须为非空有限一维序列；不自动删除缺失行")
        sample_scale = float(np.std(values))
        if not np.isfinite(sample_scale) or sample_scale <= 0:raise ValueError("训练收益没有有效变化，不能估计非退化随机模型")
        scale = sample_scale if numeric_scale is None else float(numeric_scale)
        if not np.isfinite(scale) or scale <= 0 or not np.isfinite(scale * scale) or scale * scale == 0:
            raise ValueError("numeric_scale 必须为可表示其平方的有限正数")
        scaled_returns = values / scale
        mu_start = float(np.mean(scaled_returns)) if include_mean else 0.0
        residual_seed = scaled_returns - mu_start
        if initial_variance is None:
            variance_start = (float(GARCH().backcast(residual_seed)) if initialization == "backcast"
                              else float(np.mean(residual_seed ** 2)))
        else:variance_start = float(initial_variance) / scale ** 2
        component_start = variance_start if initial_component is None else float(initial_component) / scale ** 2
        if not np.isfinite([variance_start, component_start]).all() or min(variance_start, component_start) <= 0:
            raise ValueError("初始方差与长期成分必须严格为正且有限")
        specification = {"model": model, "mean_order": (ar_order, ma_order),
            "volatility_order": (arch_order, garch_order), "asymmetric_order": int(asymmetric_order),
            "include_mean": bool(include_mean), "fractional_memory": int(fractional_memory),
            "fractional_d_bounds": d_bounds, "variance_kind": kind,
            "initialization": initialization, "numeric_scale": scale, "estimation": "gaussian_qml"}
        presample = {} if presample is None else presample
        if set(presample) - {"returns", "residuals", "variances", "components"}:raise ValueError("presample 存在未定义的历史字段")
        history = {}
        for name in ("returns", "residuals", "variances", "components"):
            past_values = np.asarray(presample.get(name, []), dtype=float)
            if past_values.ndim != 1 or not np.isfinite(past_values).all():raise ValueError(f"presample.{name} 必须为有限一维历史")
            if name in ("variances", "components") and np.any(past_values <= 0):raise ValueError(f"presample.{name} 必须严格为正")
            history[f"{name}_history"] = past_values / (scale ** 2 if name in ("variances", "components") else scale)
        raw_start, bounds = [], []
        if include_mean:raw_start.append(mu_start); bounds.append((None, None))
        raw_start.extend([0.0] * (ar_order + ma_order)); bounds.extend([(-8.0, 8.0)] * (ar_order + ma_order))
        if model == "ARFIMA-GARCH":
            d_start = 0.15 if d_bounds[0] < 0.15 < d_bounds[1] else sum(d_bounds) / 2
            proportion = (d_start - d_bounds[0]) / (d_bounds[1] - d_bounds[0])
            raw_start.append(float(np.log(proportion / (1 - proportion)))); bounds.append((-14.0, 14.0))
        omega_position = len(raw_start)
        raw_start.append(0.0 if kind == 1 else float(np.log(0.1 * variance_start)))
        bounds.append((None, None) if kind == 1 else (-30.0, 30.0))
        if kind in (0, 2):
            weights = np.r_[np.full(arch_order, 0.06 / arch_order) if arch_order else [],
                             np.full(asymmetric_order, 0.04 / asymmetric_order) if asymmetric_order else [],
                             np.full(garch_order, 0.84 / garch_order) if garch_order else []]
            raw_start.extend(np.log(weights / (1 - weights.sum())))
            bounds.extend([(-20.0, 20.0)] * len(weights))
        elif kind == 1:
            raw_start.extend([0.10] * arch_order + [-0.05] * asymmetric_order)
            bounds.extend([(None, None)] * (arch_order + asymmetric_order))
            if garch_order:
                stable_beta = np.full(garch_order, 0.85 / garch_order)
                raw_start.extend(np.arcsinh(unconstrain_stationary_univariate(stable_beta)))
                bounds.extend([(-8.0, 8.0)] * garch_order)
        elif kind == 3:
            d_start = 0.3 if d_bounds[0] < 0.3 < d_bounds[1] else sum(d_bounds) / 2
            proportion = (d_start - d_bounds[0]) / (d_bounds[1] - d_bounds[0])
            raw_start.append(float(np.log(proportion / (1 - proportion)))); bounds.append((-14.0, 14.0))
            if arch_order:raw_start.append(0.0); bounds.append((-20.0, 20.0))
            if garch_order:raw_start.append(float(np.log(0.6 / 0.4))); bounds.append((-20.0, 20.0))
        else:
            rho_start = 0.97
            raw_start.extend([np.log(rho_start / (1 - rho_start)), np.log(0.02 / 0.98)])
            bounds.extend([(-14.0, 14.0), (-20.0, 20.0)])
            weights = np.r_[np.full(arch_order, 0.08 / arch_order), np.full(garch_order, 0.83 / garch_order)] / rho_start
            raw_start.extend(np.log(weights / (1 - weights.sum())))
            bounds.extend([(-20.0, 20.0)] * len(weights))
        raw_start = np.asarray(raw_start, dtype=float)
        if len(values) <= max(len(raw_start), ar_order, ma_order, arch_order, garch_order, asymmetric_order):
            raise ValueError("训练观测数须超过自由参数数与各阶数")
        seed_coefficients = _decode_return_parameters(raw_start, specification)
        if kind in (0, 2):
            persistence = seed_coefficients["alpha"].sum() + 0.5 * seed_coefficients["gamma"].sum() + seed_coefficients["beta"].sum()
            omega_start = (1 - persistence) * variance_start
        elif kind == 1:omega_start = (1 - seed_coefficients["beta"].sum()) * np.log(variance_start)
        elif kind == 3:
            beta_value = seed_coefficients["beta"][0] if garch_order else 0.0
            omega_start = (1 - beta_value) * (1 - seed_coefficients["figarch_weights"].sum()) * variance_start
        else:omega_start = (1 - seed_coefficients["rho"]) * variance_start
        raw_start[omega_position] = omega_start if kind == 1 else np.log(max(omega_start, 1.e-12))
        seed_state = {"specification": specification, "initial_variance": variance_start,
                      "initial_component": component_start, **history}

        def objective(raw_parameters):
            candidate = {**seed_state, **_decode_return_parameters(raw_parameters, specification)}
            path = _evaluate_return_sequence(scaled_returns, candidate)
            if path["status"] != "ok":return 1.e12
            with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
                score = 0.5 * np.mean(np.log(2 * np.pi) + np.log(path["variance"]) + path["residual"] ** 2 / path["variance"])
            return float(score) if np.isfinite(score) else 1.e12

        initial_objective = objective(raw_start)
        options = {"maxiter": 500, "maxfun": 100000, "ftol": 1.e-10, "gtol": 1.e-5, "maxls": 40}
        if optimizer_options is not None:options.update(optimizer_options)
        optimization = minimize(objective, raw_start, method="L-BFGS-B", bounds=bounds, options=options)
        diagnostics = {"method": "L-BFGS-B", "success": bool(optimization.success),
            "message": str(optimization.message), "iterations": int(optimization.nit),
            "evaluations": int(optimization.nfev), "initial_average_qml": initial_objective,
            "final_average_qml": float(optimization.fun), "joint_parameters": len(raw_start)}
        if not optimization.success:
            return {**failure, "reason": f"联合优化未收敛: {optimization.message}", "optimization": diagnostics,
                    "specification": specification}
        coefficients = _decode_return_parameters(optimization.x, specification)
        # 线性换单位仅用于数值优化；所有正式系数与状态回到输入尺度。
        coefficients["mu"] *= scale
        if include_mean:coefficients["parameters"]["mean.mu"] = coefficients["mu"]
        if kind == 1:coefficients["omega"] += (1 - coefficients["beta"].sum()) * 2 * np.log(scale)
        else:coefficients["omega"] *= scale ** 2
        coefficients["parameters"]["variance.omega"] = coefficients["omega"]
        initial_state = {"status": "ok", "specification": specification, **coefficients,
            "initial_variance": variance_start * scale ** 2, "initial_component": component_start * scale ** 2,
            **{name: np.asarray(presample.get(name.removesuffix("_history"), []), dtype=float).copy() for name in history}}
        path = _evaluate_return_sequence(values, initial_state)
        if path["status"] != "ok":return {**failure, "reason": path["reason"], "optimization": diagnostics}
        history_limit = max(len(coefficients["mean_filter"]) - 1, ma_order, arch_order, garch_order,
                            asymmetric_order, fractional_memory if kind == 3 else 0, 1)
        state = deepcopy(initial_state)
        for name, realized_values in (("returns", values), ("residuals", path["residual"]), ("variances", path["variance"])):
            state[f"{name}_history"] = np.r_[initial_state[f"{name}_history"], realized_values][-history_limit:].copy()
        state["components_history"] = (np.r_[initial_state["components_history"], path["long_term_variance"]][-history_limit:].copy()
                                       if kind == 4 else np.empty(0))
        state["history_limit"], state["observations_seen"] = history_limit, len(values)
        forecast = predict_return_model(state)
        if forecast["status"] != "ok":return {**failure, "reason": forecast["reason"], "optimization": diagnostics}
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            standardized_innovations = path["residual"] / np.sqrt(path["variance"])
            loglikelihood = -float(0.5 * np.sum(np.log(2 * np.pi) + np.log(path["variance"]) + standardized_innovations ** 2))
        if not np.isfinite(standardized_innovations).all() or not np.isfinite(loglikelihood):
            return {**failure, "reason": "标准化创新或原尺度 QML 值非有限", "optimization": diagnostics}
        ar_values = [coefficients["parameters"][f"mean.ar[{lag}]"] for lag in range(1, ar_order + 1)]
        for polynomial in (np.r_[1.0, -np.asarray(ar_values)], np.r_[1.0, coefficients["ma"]],
                           np.r_[1.0, -coefficients["beta"]] if kind == 1 else np.ones(1)):
            if np.any(np.abs(np.polynomial.polynomial.polyroots(polynomial)) <= 1):
                return {**failure, "reason": "拟合系数的数值根不在平稳／可逆范围", "optimization": diagnostics}
        in_sample_df = pd.DataFrame({"observation": values, "conditional_mean": path["mean"],
            "innovation": path["residual"], "conditional_variance": path["variance"],
            "standardized_innovation": standardized_innovations}, index=observation_index)
        if kind == 4:
            in_sample_df["long_term_variance"] = path["long_term_variance"]
            in_sample_df["short_term_deviation"] = path["variance"] - path["long_term_variance"]
        return {"status": "ok", "reason": "", "parameters": coefficients["parameters"].copy(),
                "specification": specification.copy(), "state": state, "in_sample": in_sample_df,
                "optimization": diagnostics, "loglikelihood": loglikelihood, "next_forecast": forecast}
    except (ValueError, TypeError, FloatingPointError, OverflowError) as error:
        return {**failure, "reason": str(error)}


def predict_return_model(state: dict) -> dict:
    """下一采样步的条件均值／创新方差；预测不改变状态。"""
    if not isinstance(state, dict) or state.get("status") != "ok":
        return {"status": "failed", "reason": "预测需要成功拟合或更新后的有效 state"}
    path = _evaluate_return_sequence(np.array([0.0]), state)
    if path["status"] != "ok":return {"status": "failed", "reason": path["reason"]}
    forecast = {"status": "ok", "reason": "", "mean_forecast": float(path["mean"][0]),
                "variance_forecast": float(path["variance"][0])}
    if state["specification"]["variance_kind"] == 4:
        forecast["long_term_variance_forecast"] = float(path["long_term_variance"][0])
        forecast["short_term_deviation_forecast"] = forecast["variance_forecast"] - forecast["long_term_variance_forecast"]
    return forecast


def update_return_model(state: dict, observed_return: float) -> dict:
    """吸收一个实现收益，返回新状态；不重新估计系数，也不修改旧状态。"""
    if not isinstance(state, dict) or state.get("status") != "ok":
        return {"status": "failed", "reason": "更新需要成功拟合或更新后的有效 state", "state": None}
    try:observed_return = float(observed_return)
    except (ValueError, TypeError):
        return {"status": "failed", "reason": "实现收益必须为一个有限数值", "state": None}
    if not np.isfinite(observed_return):
        return {"status": "failed", "reason": "实现收益缺失或非有限；旧状态未推进", "state": None}
    path = _evaluate_return_sequence(np.array([observed_return]), state)
    if path["status"] != "ok":return {"status": "failed", "reason": path["reason"], "state": None}
    updated_state = deepcopy(state)
    for name, value in (("returns", observed_return), ("residuals", path["residual"][0]), ("variances", path["variance"][0])):
        updated_state[f"{name}_history"] = np.r_[state[f"{name}_history"], value][-state["history_limit"]:]
    if state["specification"]["variance_kind"] == 4:
        updated_state["components_history"] = np.r_[state["components_history"], path["long_term_variance"][0]][-state["history_limit"]:]
    updated_state["observations_seen"] += 1
    next_forecast = predict_return_model(updated_state)
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        standardized_innovation = float(path["residual"][0] / np.sqrt(path["variance"][0]))
    if next_forecast["status"] != "ok" or not np.isfinite(standardized_innovation):
        reason = next_forecast["reason"] or "标准化创新非有限"
        return {"status": "failed", "reason": f"吸收观测后下一步无法有效预测: {reason}", "state": None}
    output = {"status": "ok", "reason": "", "state": updated_state, "next_forecast": next_forecast,
        "conditional_mean": float(path["mean"][0]), "conditional_variance": float(path["variance"][0]),
        "innovation": float(path["residual"][0]),
        "standardized_innovation": standardized_innovation}
    if state["specification"]["variance_kind"] == 4:
        output["long_term_variance"] = float(path["long_term_variance"][0])
        output["short_term_deviation"] = output["conditional_variance"] - output["long_term_variance"]
    return output

In [25]:
RETURN_MODEL_FIELD_LABELS = {
    "model": "模型", "status": "状态", "reason": "原因", "iterations": "联合优化迭代数",
    "mean_forecast": "下一步收益均值", "variance_forecast": "下一步条件方差",
    "long_term_variance_forecast": "下一步长期方差成分",
    "short_term_deviation_forecast": "下一步短期偏离",
    "observation": "实现收益", "conditional_mean": "条件均值",
    "innovation": "创新残差", "conditional_variance": "条件方差",
    "standardized_innovation": "标准化创新", "long_term_variance": "长期方差成分",
    "short_term_deviation": "短期偏离", "parameter": "具名系数", "value": "系数值",
}
RETURN_MODEL_COLUMN_LABELS = {
    name: f"{name}（{label}）" for name, label in RETURN_MODEL_FIELD_LABELS.items()
}

In [26]:
import pyarrow as pa
from copy import deepcopy
from a01_Methods.b04_ReturnVolatilityModeling.c02_ModelObservations import transform_model_input

demo_prediction_fields = [
    pa.field("sample_id", pa.int64(), nullable=False), pa.field("sample_at", pa.timestamp("us", tz="Asia/Shanghai"), nullable=False),
    pa.field("trading_date", pa.date32(), nullable=False), pa.field("sample_side", pa.string(), nullable=False),
    pa.field("training_start_sample_id", pa.int64()), pa.field("training_end_sample_id", pa.int64()),
    pa.field("training_points", pa.int64(), nullable=False), pa.field("model", pa.string(), nullable=False),
    pa.field("status", pa.string(), nullable=False), pa.field("reason", pa.string(), nullable=False),
    pa.field("mean_forecast", pa.float64()), pa.field("parameters", pa.map_(pa.string(), pa.float64())),
    pa.field("update_status", pa.string()), pa.field("update_reason", pa.string()),
    pa.field("updated_mean_forecast", pa.float64()),
]

def _compute_return_demo(demo_observations, *, history_points=500,
                         models=RETURN_MODEL_NAMES, target_positions=None):
    """逐目标重新估计；分片只选择目标，不切片训练历史；返回内存验收结果。"""
    demo_history_points = history_points
    selected_positions = np.arange(len(demo_observations), dtype=int) if target_positions is None else np.asarray(target_positions, dtype=int)
    if selected_positions.ndim != 1 or len(np.unique(selected_positions)) != len(selected_positions):
        raise ValueError("目标坐标必须是一维不重复位置")
    if len(selected_positions) and (selected_positions.min() < 0 or selected_positions.max() >= len(demo_observations)):
        raise ValueError("目标坐标超出完整输入范围")
    assert demo_observations.iloc[0]["log_return_status"] == "no_previous_sample"
    demo_inputs = transform_model_input(demo_observations, observation_col="log_return")
    demo_values = demo_inputs["model_input"].to_numpy(dtype=float, na_value=np.nan)
    demo_result_schema = pa.schema([*demo_prediction_fields,
        *[pa.field(name, pa.float64()) for name in ("variance_forecast", "long_term_variance_forecast",
          "short_term_deviation_forecast", "updated_variance_forecast")]])
    demo_results = {}
    demo_update_checks = {}
    for model_name in models:
        rows, update_checks = [], 0
        options = dict(model=model_name, mean_order=(1, 1), volatility_order=(1, 1),
            fractional_memory=30 if model_name in ("ARFIMA-GARCH", "ARMA-FIGARCH") else None)
        for completed, position in enumerate(selected_positions, 1):
            target = demo_inputs.iloc[position]
            start = max(1, position - demo_history_points)
            history = demo_values[start:position]
            row = dict(sample_id=target.sample_id, sample_at=target.sample_at, trading_date=target.trading_date,
                sample_side=target.sample_side, training_start_sample_id=demo_inputs.iloc[start].sample_id if len(history) else None,
                training_end_sample_id=demo_inputs.iloc[position - 1].sample_id if len(history) else None,
                training_points=len(history), model=model_name, parameters=None)
            fit = fit_return_model(history, **options)
            row.update(status=fit["status"], reason=fit["reason"])
            if fit["status"] == "ok":
                original_state = deepcopy(fit["state"])
                prediction = predict_return_model(fit["state"])
                row.update(status=prediction["status"], reason=prediction["reason"], parameters=fit["parameters"])
                if prediction["status"] == "ok":
                    row.update({k: v for k, v in prediction.items() if k not in ("status", "reason")})
                    np.testing.assert_allclose(prediction["mean_forecast"], fit["next_forecast"]["mean_forecast"])
                    np.testing.assert_allclose(prediction["variance_forecast"], fit["next_forecast"]["variance_forecast"])
                    update = update_return_model(fit["state"], target.model_input)
                    row.update(update_status=update["status"], update_reason=update["reason"])
                    assert fit["state"]["observations_seen"] == original_state["observations_seen"]
                    for key in ("returns_history", "residuals_history", "variances_history"):
                        np.testing.assert_array_equal(fit["state"][key], original_state[key])
                    if update["status"] == "ok":
                        assert update["state"]["parameters"] == fit["parameters"]
                        assert update["state"]["observations_seen"] == len(history) + 1
                        np.testing.assert_allclose(update["conditional_mean"], prediction["mean_forecast"])
                        np.testing.assert_allclose(update["conditional_variance"], prediction["variance_forecast"])
                        row.update(updated_mean_forecast=update["next_forecast"]["mean_forecast"],
                            updated_variance_forecast=update["next_forecast"]["variance_forecast"])
                        update_checks += 1
                    if model_name == "ARMA-Component GARCH":
                        np.testing.assert_allclose(row["variance_forecast"], row["long_term_variance_forecast"] + row["short_term_deviation_forecast"])
            rows.append(row)
            if completed % 100 == 0 or completed == len(selected_positions):
                print(f"c03 {model_name} {completed}/{len(selected_positions)} 点；成功更新检查 {update_checks}", flush=True)
        result = pd.DataFrame(rows).reindex(columns=demo_result_schema.names)
        assert len(result) == len(selected_positions)
        assert result.sample_id.tolist() == demo_inputs.iloc[selected_positions].sample_id.tolist()
        assert result.loc[result.training_end_sample_id.notna(), "training_end_sample_id"].lt(
            result.loc[result.training_end_sample_id.notna(), "sample_id"]).all()
        demo_results[model_name] = result
        demo_update_checks[model_name] = update_checks
    return dict(results=demo_results, update_checks=demo_update_checks, schema=demo_result_schema)


## 真实行情内存试算与完整云端 demo

内存试算的示例历史窗口为 30 点，完整云 demo 为 500 点。下面使用 RB 的 2024-12-31—2025-01-02 真实复权分钟行情，在 Notebook 内试算全部函数，仅展示结果，不写数据文件。完整 demo 使用上游 2010-01-01—2026-09-30 的全部可用序列，由云端调用下列同一套组合函数；每个目标点均用此前最多 500 个采样观测重新拟合，失败位置与原因保留。季节窗口 1 年／3 年、N=15、GK sigma×全合约成交额轴等是方法 demo 示例参数，正式大实验配置另行定义。


In [27]:
from pathlib import Path
import sys
from IPython.display import display

project_markers = [".git", ".env", "config/settings.py"]
current_path = Path.cwd().resolve()
for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        sys.path.insert(0, str(candidate_root / "04_Research"))
        break
else:
    raise RuntimeError("未找到项目根目录")
from config.settings import settings
from config.data_contracts import FUTURES_MINUTE_SCHEMA

# 本地仅做真实小范围内存试算，不写 demo 数据文件。
demo_variety = "RB"
demo_source_range = "train20100101-20241231_test20250101-20260930"
demo_start_date = pd.Timestamp("2024-12-31")
demo_end_date = pd.Timestamp("2025-01-02")
demo_test_start = pd.Timestamp("2025-01-01")
demo_sampling = "N15_gk_money_all_contracts_log_interpolate_fixed"
demo_base = candidate_root / "04_Research/a01_Methods"
demo_modeling = demo_base / "b04_ReturnVolatilityModeling"
demo_window_names = ("1y", "3y")
demo_history_points = 30  # 小范围内存试算；完整云 demo 固定使用 500 点。
# 完整 demo 由显式云入口执行；此范围不改变正式大实验截至 2026-06-30 的 scope。
demo_cloud_scope = {"start": "2010-01-01", "end": "2026-09-30", "test_start": "2025-01-01"}


In [28]:
from a01_Methods.b04_ReturnVolatilityModeling.c01_IntradaySeasonality import (
    _load_intraday_demo_inputs, _compute_intraday_demo)
from a01_Methods.b04_ReturnVolatilityModeling.c02_ModelObservations import _compute_observations_demo, _load_demo_sampling_axis
if "_b04_observations_preview" not in globals():
    if "_b04_intraday_preview" not in globals():
        demo_real_inputs = _load_intraday_demo_inputs(candidate_root, start=demo_start_date,
            end=demo_end_date, test_start=demo_test_start)
        _b04_intraday_preview = _compute_intraday_demo(demo_real_inputs["main"], demo_real_inputs["calendar"],
            start=demo_start_date, test_start=demo_test_start)
    demo_sampling_axis = _load_demo_sampling_axis(candidate_root, start=demo_start_date, end=demo_end_date)
    _b04_observations_preview = _compute_observations_demo(_b04_intraday_preview["contributions"],
        _b04_intraday_preview["seasonality"], test_start=demo_test_start, sampling_axis=demo_sampling_axis)
demo_observations = _b04_observations_preview["observation_sets"]["none"]


In [29]:
demo_prediction_labels = {"sample_id": "sample_id（采样编号）", "sample_at": "sample_at（预测目标时刻）",
    "trading_date": "trading_date（交易日）", "sample_side": "sample_side（采样侧）",
    "training_start_sample_id": "training_start_sample_id（训练首个编号）",
    "training_end_sample_id": "training_end_sample_id（训练末个编号）", "training_points": "training_points（训练观测数）",
    "parameters": "parameters（具名拟合系数）", "update_status": "update_status（更新状态）",
    "update_reason": "update_reason（更新原因）", "updated_mean_forecast": "updated_mean_forecast（更新后下一步预测）",
    "features": "features（HAR 窗口均值）", "components": "components（预测成分）"}


每个目标使用完整输入中此前最多 500 个采样观测重新拟合；初始窗口未满时使用已有历史，首个锚点及拟合失败也保留。云端节点只分配目标坐标，不将每个节点的稀疏点集作为训练序列。Notebook 仅试算所选小区间，完整上游区间由云端验收。


In [30]:
demo_prediction_preview = _compute_return_demo(demo_observations, history_points=demo_history_points)
demo_results, demo_update_checks = demo_prediction_preview["results"], demo_prediction_preview["update_checks"]
assert all(count > 0 for count in demo_update_checks.values()), demo_update_checks
for case, result in demo_results.items():
    print(f"{case}：成功更新检查 {demo_update_checks[case]}", flush=True)
    display(result.groupby(["status", "reason"]).size().rename("count（采样点数）").rename_axis(
        ["status（预测状态）", "reason（失败原因）"]).to_frame())
display(demo_results["ARMA-Component GARCH"].tail().rename(
    columns={**RETURN_MODEL_COLUMN_LABELS, **demo_prediction_labels,
             "updated_variance_forecast": "updated_variance_forecast（更新后下一步条件方差）"}).rename_axis("row（采样行）"))
print("c03 真实小范围逐点试算完成；未写数据文件", flush=True)


c03 ARMA-GARCH 69/69 点；成功更新检查 34


c03 ARMA-EGARCH 69/69 点；成功更新检查 33


c03 ARMA-GJR-GARCH 69/69 点；成功更新检查 34


c03 ARMA-FIGARCH 69/69 点；成功更新检查 34


c03 ARMA-Component GARCH 69/69 点；成功更新检查 34


...

,,count（采样点数）
status（预测状态）,reason（失败原因）,
failed,observations 必须为非空有限一维序列；不自动删除缺失行,35
ok,,34


ARFIMA-GARCH：成功更新检查 34


,,count（采样点数）
status（预测状态）,reason（失败原因）,
failed,observations 必须为非空有限一维序列；不自动删除缺失行,35
ok,,34


,sample_id（采样编号）,sample_at（预测目标时刻）,trading_date（交易日）,sample_side（采样侧）,training_start_sample_id（训练首个编号）,training_end_sample_id（训练末个编号）,training_points（训练观测数）,model（模型）,status（状态）,reason（原因）,mean_forecast（下一步收益均值）,parameters（具名拟合系数）,update_status（更新状态）,update_reason（更新原因）,updated_mean_forecast（更新后下一步预测）,variance_forecast（下一步条件方差）,long_term_variance_forecast（下一步长期方差成分）,short_term_deviation_forecast（下一步短期偏离）,updated_variance_forecast（更新后下一步条件方差）
row（采样行）,,,,,,,,,,,,,,,,,,,
64,65,2025-01-02 14:04:19.119334+08:00,2025-01-02,test,35.0,64.0,30,ARMA-Component GARCH,ok,,-0.000181,"{'mean.mu': 1.2558552425678856e-05, 'mean.ar[1]': 0.7134541956114173, 'mean.ma[1]': -0.38391149351207216, 'variance.omega': 1.324529481491588e-13, 'variance.rho': 0.9819186501400262, 'variance.eta': 2.0238851743330172e-09, 'variance.alpha[1]': 4.171538267074372e-18, 'variance.beta[1]': 0.9819186481161409}",ok,,-0.000035,6.418602e-07,6.418602e-07,0.000000e+00,6.302546e-07
65,66,2025-01-02 14:15:26.888087+08:00,2025-01-02,test,36.0,65.0,30,ARMA-Component GARCH,ok,,0.000010,"{'mean.mu': 3.744248768908753e-05, 'mean.ar[1]': 0.6577340265826266, 'mean.ma[1]': -0.31254474898769735, 'variance.omega': 6.101367542048193e-14, 'variance.rho': 0.9752759384445843, 'variance.eta': 2.0101935251155835e-09, 'variance.alpha[1]': 4.143317674634533e-18, 'variance.beta[1]': 0.9752759364343907}",ok,,-0.000737,5.438496e-07,5.438496e-07,0.000000e+00,5.304035e-07
66,67,2025-01-02 14:24:14.002070+08:00,2025-01-02,test,37.0,66.0,30,ARMA-Component GARCH,ok,,-0.000762,"{'mean.mu': -0.00019411740362126567, 'mean.ar[1]': 0.8660398175526902, 'mean.ma[1]': -0.6003676290706492, 'variance.omega': 3.3473109756529895e-08, 'variance.rho': 0.9517929003514811, 'variance.eta': 3.406684394812192e-07, 'variance.alpha[1]': 2.0998584773948424e-11, 'variance.beta[1]': 0.9517851982929161}",ok,,-0.000436,8.297604e-07,8.297604e-07,1.075817e-17,8.232332e-07
67,68,2025-01-02 14:42:16.490935+08:00,2025-01-02,test,38.0,67.0,30,ARMA-Component GARCH,ok,,-0.000367,"{'mean.mu': -6.636538767015023e-05, 'mean.ar[1]': 0.8506360849990975, 'mean.ma[1]': -0.6285291847867476, 'variance.omega': 5.00793256433328e-08, 'variance.rho': 0.9327835622896365, 'variance.eta': 4.0058494071843573e-08, 'variance.alpha[1]': 1.4817499487110495e-13, 'variance.beta[1]': 0.9327829193422581}",ok,,-0.000116,8.179110e-07,8.179110e-07,1.031263e-19,8.130133e-07
68,69,2025-01-02 14:58:11.698689+08:00,2025-01-02,test,39.0,68.0,30,ARMA-Component GARCH,ok,,-0.000059,"{'mean.mu': 3.916234449653399e-05, 'mean.ar[1]': 0.8287065083340562, 'mean.ma[1]': -0.6212773138945707, 'variance.omega': 7.311032176897706e-08, 'variance.rho': 0.9071446356040997, 'variance.eta': 5.131714412404114e-07, 'variance.alpha[1]': 8.338209141411282e-18, 'variance.beta[1]': 0.9071446315586907}",ok,,0.000049,8.184223e-07,8.184223e-07,0.000000e+00,8.155374e-07


c03 真实小范围逐点试算完成；未写数据文件
